# CLEAR: Live Webcam Detection & Tracking Test (`Person` + `Tennis Racket` + `Sports Ball`)

Quick interactive notebook to test real-time detection and tracking of:
- **Person** (`COCO class 0`)
- **Tennis Racket** (`COCO class 38`)
- **Tennis Ball** (`COCO class 32` + high-speed optic-yellow HSV / 3-frame motion fallback)

Supports:
1. **Local Webcam (`cv2.VideoCapture(0)`)** — Live OpenCV window or inline Jupyter stream.
2. **Google Colab Webcam** — Browser JS `getUserMedia` stream to Colab GPU.
3. **Racket-Ball Hit Detector** — Flashes `HIT / CONTACT` when ball trajectory intersects racket box.

In [ ]:
# @title 1. Setup & Dependencies
import subprocess
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "opencv-python-headless", "numpy", "matplotlib"])

import time
from collections import deque
from dataclasses import dataclass, field
from typing import Deque, Dict, List, Optional, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np

# Try loading Ultralytics YOLO (COCO classes: 0=person, 32=sports ball, 38=tennis racket)
YOLO_AVAILABLE = False
yolo_model = None
try:
    from ultralytics import YOLO
    yolo_model = YOLO("yolov8n.pt")
    YOLO_AVAILABLE = True
    print("Loaded YOLOv8n (COCO classes: 0=person, 32=sports ball, 38=tennis racket)")
except Exception as e:
    print(f"Ultralytics YOLO not active ({e}). Using OpenCV Haar/HOG + Optic-Yellow HSV + Racket Contour fallback.")

In [ ]:
# @title 2. Detection & Multi-Object Tracker Configuration
TARGET_CLASSES = {
    0: ("PERSON", (255, 190, 40)),         # Cyan-Blue
    32: ("TENNIS BALL", (40, 245, 255)),   # Optic Yellow
    38: ("TENNIS RACKET", (210, 95, 255)), # Magenta-Purple
}

CONF_PERSON = 0.35  # @param {type:"slider", min:0.15, max:0.80, step:0.05}
CONF_RACKET = 0.25  # @param {type:"slider", min:0.10, max:0.75, step:0.05}
CONF_BALL = 0.18    # @param {type:"slider", min:0.05, max:0.70, step:0.05}
ENABLE_HSV_BALL_FALLBACK = True  # @param {type:"boolean"}
BALL_HSV_LOWER = (22, 85, 120)
BALL_HSV_UPPER = (46, 255, 255)
TRAIL_LENGTH = 32

@dataclass
class TrackedObject:
    track_id: int
    cls_id: int
    label: str
    bbox: Tuple[float, float, float, float]  # x1, y1, x2, y2
    center: Tuple[float, float]
    velocity: Tuple[float, float] = (0.0, 0.0)
    conf: float = 0.0
    missed: int = 0
    trail: Deque[Tuple[int, int]] = field(default_factory=lambda: deque(maxlen=TRAIL_LENGTH))

class MultiClassTracker:
    """Tracks Person (0), Tennis Ball (32), and Tennis Racket (38) with Kalman/EMA smoothing & trails."""

    def __init__(self, max_missed: int = 10, alpha: float = 0.55) -> None:
        self.max_missed = max_missed
        self.alpha = alpha
        self.next_id = 1
        self.tracks: Dict[int, TrackedObject] = {}
        self.gray_buf: Deque[np.ndarray] = deque(maxlen=3)
        self.hog = cv2.HOGDescriptor()
        self.hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())
        self.hit_flash_frames = 0

    def detect_and_track(self, frame_bgr: np.ndarray) -> Tuple[np.ndarray, Dict[int, TrackedObject]]:
        t0 = time.perf_counter()
        raw_dets = self._detect(frame_bgr)
        self._associate(raw_dets)
        fps = 1.0 / max(1e-4, time.perf_counter() - t0)
        vis = self._render_hud(frame_bgr, fps)
        return vis, self.tracks

    def _detect(self, frame_bgr: np.ndarray) -> List[Tuple[int, Tuple[float, float, float, float], float]]:
        dets: List[Tuple[int, Tuple[float, float, float, float], float]] = []
        h, w = frame_bgr.shape[:2]

        # 1. Primary YOLOv8 detector for Person (0), Sports Ball (32), Tennis Racket (38)
        if YOLO_AVAILABLE and yolo_model is not None:
            results = yolo_model.predict(frame_bgr, classes=[0, 32, 38], conf=0.15, verbose=False)
            if results and len(results) > 0:
                for box in results[0].boxes:
                    cls_id = int(box.cls[0].item())
                    conf = float(box.conf[0].item())
                    if cls_id == 0 and conf < CONF_PERSON:
                        continue
                    if cls_id == 38 and conf < CONF_RACKET:
                        continue
                    if cls_id == 32 and conf < CONF_BALL:
                        continue
                    x1, y1, x2, y2 = [float(v) for v in box.xyxy[0].tolist()]
                    dets.append((cls_id, (x1, y1, x2, y2), conf))

        # 2. Fast Optic-Yellow Ball Fallback (catches motion-blurred balls missed by YOLO)
        has_ball = any(c == 32 for c, _, _ in dets)
        if ENABLE_HSV_BALL_FALLBACK and not has_ball:
            gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
            self.gray_buf.append(gray)
            hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
            mask = cv2.inRange(hsv, np.array(BALL_HSV_LOWER, dtype=np.uint8), np.array(BALL_HSV_UPPER, dtype=np.uint8))
            mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in contours:
                area = cv2.contourArea(cnt)
                if area < 25 or area > (w * h * 0.08):
                    continue
                (cx, cy), r = cv2.minEnclosingCircle(cnt)
                perim = max(1.0, cv2.arcLength(cnt, True))
                circ = 4.0 * np.pi * area / (perim * perim)
                if circ >= 0.42 and 4 <= r <= 95:
                    dets.append((32, (cx - r, cy - r, cx + r, cy + r), round(float(min(0.92, circ)), 2)))
                    break

        # 3. Classical fallback if YOLO is unavailable (Person HOG/Silhouette + Racket oval contour)
        if not YOLO_AVAILABLE:
            rects, weights = self.hog.detectMultiScale(frame_bgr, winStride=(8, 8), padding=(8, 8), scale=1.08)
            for (x, y, bw, bh), wt in zip(rects, weights):
                dets.append((0, (float(x), float(y), float(x + bw), float(y + bh)), float(min(0.95, wt))))

            # Silhouette fallback if HOG misses close-up upper-body webcam person
            if not any(c == 0 for c, _, _ in dets):
                hsv_f = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
                p_mask = cv2.inRange(hsv_f, np.array([95, 80, 80], dtype=np.uint8), np.array([130, 255, 255], dtype=np.uint8))
                p_cnts, _ = cv2.findContours(p_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for cnt in p_cnts:
                    if cv2.contourArea(cnt) > 4000:
                        px, py, pw, ph = cv2.boundingRect(cnt)
                        dets.append((0, (float(px), float(max(0, py - 65)), float(px + pw), float(py + ph)), 0.82))
                        break

            # Detect racket-head oval / magenta or dark frame contour
            edges = cv2.Canny(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY), 60, 160)
            r_cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in r_cnts:
                if len(cnt) >= 5 and 900 < cv2.contourArea(cnt) < 28000:
                    (ex, ey), (MA, ma), angle = cv2.fitEllipse(cnt)
                    if 1.15 <= max(MA, ma) / max(1.0, min(MA, ma)) <= 1.85:
                        rx, ry, rw, rh = cv2.boundingRect(cnt)
                        dets.append((38, (float(rx), float(ry), float(rx + rw), float(ry + rh)), 0.78))
                        break

        return dets

    def _associate(self, raw_dets: List[Tuple[int, Tuple[float, float, float, float], float]]) -> None:
        for trk in self.tracks.values():
            trk.missed += 1

        for cls_id, bbox, conf in raw_dets:
            cx = 0.5 * (bbox[0] + bbox[2])
            cy = 0.5 * (bbox[1] + bbox[3])
            # Find closest existing track of the same class
            same_cls = [t for t in self.tracks.values() if t.cls_id == cls_id]
            best_trk: Optional[TrackedObject] = None
            best_dist = 180.0
            for cand in same_cls:
                d = float(np.hypot(cx - cand.center[0], cy - cand.center[1]))
                if d < best_dist:
                    best_dist = d
                    best_trk = cand

            if best_trk is not None:
                a = self.alpha
                ob = best_trk.bbox
                sb = (
                    (1 - a) * ob[0] + a * bbox[0],
                    (1 - a) * ob[1] + a * bbox[1],
                    (1 - a) * ob[2] + a * bbox[2],
                    (1 - a) * ob[3] + a * bbox[3],
                )
                nc = (0.5 * (sb[0] + sb[2]), 0.5 * (sb[1] + sb[3]))
                best_trk.velocity = (nc[0] - best_trk.center[0], nc[1] - best_trk.center[1])
                best_trk.bbox = sb
                best_trk.center = nc
                best_trk.conf = conf
                best_trk.missed = 0
                best_trk.trail.append((int(round(nc[0])), int(round(nc[1]))))
            else:
                label_str = TARGET_CLASSES.get(cls_id, ("OBJECT", (255, 255, 255)))[0]
                new_trk = TrackedObject(
                    track_id=self.next_id,
                    cls_id=cls_id,
                    label=label_str,
                    bbox=bbox,
                    center=(cx, cy),
                    conf=conf,
                )
                new_trk.trail.append((int(round(cx)), int(round(cy))))
                self.tracks[self.next_id] = new_trk
                self.next_id += 1

        # Prune stale tracks
        stale = [tid for tid, t in self.tracks.items() if t.missed > self.max_missed]
        for tid in stale:
            del self.tracks[tid]

        # Check Ball-Racket Contact / Hit
        balls = [t for t in self.tracks.values() if t.cls_id == 32 and t.missed <= 2]
        rackets = [t for t in self.tracks.values() if t.cls_id == 38 and t.missed <= 2]
        for b in balls:
            for r in rackets:
                rx1, ry1, rx2, ry2 = r.bbox
                pad = 25.0
                if (rx1 - pad <= b.center[0] <= rx2 + pad) and (ry1 - pad <= b.center[1] <= ry2 + pad):
                    self.hit_flash_frames = 12

    def _render_hud(self, frame_bgr: np.ndarray, fps: float) -> np.ndarray:
        canvas = frame_bgr.copy()
        for trk in self.tracks.values():
            if trk.missed > 3:
                continue
            _, color = TARGET_CLASSES.get(trk.cls_id, ("OBJ", (200, 200, 200)))
            x1, y1, x2, y2 = [int(round(v)) for v in trk.bbox]

            # Draw motion trail for Ball (32) and Racket (38)
            if trk.cls_id in (32, 38) and len(trk.trail) >= 2:
                pts = list(trk.trail)
                for i in range(1, len(pts)):
                    thickness = max(1, int(round((i / len(pts)) * 5)))
                    cv2.line(canvas, pts[i - 1], pts[i], color, thickness, cv2.LINE_AA)

            # Draw bounding box & velocity vector
            cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2, cv2.LINE_AA)
            cx, cy = int(round(trk.center[0])), int(round(trk.center[1]))
            vx, vy = trk.velocity
            if np.hypot(vx, vy) > 1.5:
                cv2.arrowedLine(
                    canvas,
                    (cx, cy),
                    (int(round(cx + vx * 3.5)), int(round(cy + vy * 3.5))),
                    color,
                    2,
                    tipLength=0.3,
                )

            spd_px = float(np.hypot(vx, vy))
            tag = f"#{trk.track_id} {trk.label} ({trk.conf:.2f}) | {spd_px:.1f}px/f"
            (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.50, 1)
            ty = max(th + 8, y1 - 6)
            cv2.rectangle(canvas, (x1, ty - th - 6), (x1 + tw + 8, ty + 4), (15, 20, 28), -1)
            cv2.putText(canvas, tag, (x1 + 4, ty - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.50, color, 1, cv2.LINE_AA)

        # Top telemetry bar
        n_person = sum(1 for t in self.tracks.values() if t.cls_id == 0 and t.missed <= 2)
        n_racket = sum(1 for t in self.tracks.values() if t.cls_id == 38 and t.missed <= 2)
        n_ball = sum(1 for t in self.tracks.values() if t.cls_id == 32 and t.missed <= 2)
        cv2.rectangle(canvas, (12, 12), (560, 52), (15, 20, 28), -1)
        cv2.rectangle(canvas, (12, 12), (560, 52), (80, 140, 220), 1)
        status = f"FPS: {fps:4.1f} | PERSON: {n_person} | RACKET: {n_racket} | BALL: {n_ball}"
        cv2.putText(canvas, status, (22, 38), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (240, 248, 255), 2, cv2.LINE_AA)

        if self.hit_flash_frames > 0:
            self.hit_flash_frames -= 1
            cv2.rectangle(canvas, (575, 12), (830, 52), (35, 195, 85), -1)
            cv2.putText(canvas, "RACKET-BALL CONTACT!", (588, 38), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (255, 255, 255), 2, cv2.LINE_AA)

        return canvas

tracker = MultiClassTracker()
print("MultiClassTracker initialized (Person + Tennis Racket + Tennis Ball).")

In [ ]:
# @title 3. Quick Synthetic Sanity Check (Verifies Person + Racket + Ball Tracking Immediately)
def make_synthetic_webcam_test_frame(step: int, width: int = 960, height: int = 540) -> np.ndarray:
    frame = np.full((height, width, 3), (38, 46, 58), dtype=np.uint8)
    # Person silhouette
    px = int(width * 0.35 + 25 * np.sin(step * 0.08))
    py = int(height * 0.58)
    cv2.rectangle(frame, (px - 45, py - 110), (px + 45, py + 140), (195, 130, 55), -1)  # Torso
    cv2.circle(frame, (px, py - 145), 32, (130, 160, 200), -1)  # Head

    # Tennis Racket (handle + oval head)
    rx = px + 95 + int(20 * np.cos(step * 0.15))
    ry = py - 50 + int(15 * np.sin(step * 0.15))
    cv2.line(frame, (px + 40, py - 20), (rx - 30, ry + 20), (180, 180, 190), 6, cv2.LINE_AA)
    cv2.ellipse(frame, (rx, ry), (46, 32), -25, 0, 360, (220, 90, 240), 4, cv2.LINE_AA)

    # Optic-yellow Tennis Ball flying toward/away from racket
    bx = int(rx + 30 + 180 * abs(np.cos(step * 0.12)))
    by = int(ry - 40 * np.sin(step * 0.24))
    cv2.circle(frame, (bx, by), 16, (38, 240, 248), -1, cv2.LINE_AA)
    return frame

demo_tracker = MultiClassTracker()
vis_frame = None
for s in range(18):
    test_f = make_synthetic_webcam_test_frame(s)
    vis_frame, _ = demo_tracker.detect_and_track(test_f)

plt.figure(figsize=(11, 6), facecolor="#0f172a")
plt.imshow(cv2.cvtColor(vis_frame, cv2.COLOR_BGR2RGB))
plt.title("Sanity Check: Tracked Ball Trail + Racket + Person HUD", color="white")
plt.axis("off")
plt.show()

In [ ]:
# @title 4A. Run Live Local Webcam (`cv2.VideoCapture(0)`) — Press 'q' to Stop
# Run this cell when running Jupyter locally on your laptop/desktop.
CAMERA_INDEX = 0  # @param {type:"integer"}
MAX_FRAMES = 600  # @param {type:"integer"}

if not IN_COLAB:
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print(f"Could not open webcam index {CAMERA_INDEX}. Check camera permissions.")
    else:
        print("Webcam active! Hold up a tennis ball and racket. Press 'q' in the OpenCV window to stop.")
        live_tracker = MultiClassTracker()
        frame_count = 0
        while frame_count < MAX_FRAMES:
            ret, frame = cap.read()
            if not ret:
                break
            annotated, _ = live_tracker.detect_and_track(frame)
            cv2.imshow("CLEAR Webcam Test: Person + Racket + Ball (Press Q to quit)", annotated)
            if (cv2.waitKey(1) & 0xFF) == ord('q'):
                break
            frame_count += 1
        cap.release()
        cv2.destroyAllWindows()

In [ ]:
# @title 4B. Run Live Webcam in Google Colab (Browser JS Stream -> GPU YOLO Tracker)
# Run this cell when using Google Colab in your browser!
if IN_COLAB:
    from base64 import b64decode, b64encode
    from google.colab.output import eval_js
    from IPython.display import HTML, display

    def start_colab_webcam():
        js = HTML('''
        <div style="display:flex; flex-direction:column; align-items:center; gap:8px;">
          <video id="colabCam" style="display:none;" playsinline autoplay></video>
          <img id="colabOut" style="border:2px solid #38bdf8; border-radius:8px; max-width:100%;" />
          <button id="stopCamBtn" style="padding:8px 16px; background:#ef4444; color:white; border:none; border-radius:6px; font-weight:bold; cursor:pointer;">Stop Webcam</button>
        </div>
        <script>
          var video = document.getElementById('colabCam');
          var outImg = document.getElementById('colabOut');
          var stopBtn = document.getElementById('stopCamBtn');
          var stream = null;
          var stopped = false;
          stopBtn.onclick = () => { stopped = true; if (stream) stream.getTracks().forEach(t => t.stop()); };

          async function initCam() {
            stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
            video.srcObject = stream;
            await new Promise(resolve => video.onloadedmetadata = resolve);
          }
          async function grabFrame(prevB64) {
            if (prevB64) outImg.src = prevB64;
            if (stopped) return '';
            if (!stream) await initCam();
            var canvas = document.createElement('canvas');
            canvas.width = video.videoWidth || 640;
            canvas.height = video.videoHeight || 480;
            canvas.getContext('2d').drawImage(video, 0, 0);
            return canvas.toDataURL('image/jpeg', 0.8);
          }
        </script>
        ''')
        display(js)

    start_colab_webcam()
    colab_tracker = MultiClassTracker()
    prev_overlay_b64 = ""
    while True:
        data_url = eval_js(f'grabFrame("{prev_overlay_b64}")')
        if not data_url:
            print("Webcam stream stopped.")
            break
        img_bytes = b64decode(data_url.split(',')[1])
        arr = np.frombuffer(img_bytes, dtype=np.uint8)
        frame = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        vis, _ = colab_tracker.detect_and_track(frame)
        _, enc = cv2.imencode('.jpg', vis, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
        prev_overlay_b64 = "data:image/jpeg;base64," + b64encode(enc).decode('ascii')